We'll use pint for units:

In [1]:
from pint import UnitRegistry

u = UnitRegistry()

Emissions cost example: wind vs solar
-------------------------------------
*Note: This example does not use the same data as the exercise, though both are representative of actual values for different installations. In the exercise, use the numbers you calculate or are given there.*

What is **embodied carbon per MW of installed capacity** ($\mathrm{EIC}\_{\text{mat}}$) for **(a) wind** and **(b) solar**? 

Use the following materials requirements (MI) for wind and solar:

| material | MI for wind / \[t/MW\] | MI for solar / [\t/MW\] | 
|----------|------------------------|-------------------------|
|concrete  | 500                    | 500                     | 
|steel     | 50                     | 0                       | 
|aluminium | 0                      | 60                      | 
|copper    | 5                      | 4                       | 
|silicon   | 0                      | 5                       | 

And the following characterization factors:

| material | CF / \[t CO2 / t\] | 
|----------|--------------------|
|concrete  | 0.18               |
|steel     | 1.9                |
|aluminium | 6.8                |
|copper    | 4.6                |
|silicon   | 10                 |

In [2]:
CF = {"concrete": 0.18, "steel": 1.9, "aluminium": 6.8, "copper": 4.6, "silicon": 10}

MI_wind = {"concrete": 500 * u("t/MW"), "steel": 100 * u("t/MW"), "copper": 5 * u("t/MW")}

MI_solar = {"concrete": 500 * u("t/MW"), "aluminium": 60 * u("t/MW"), "copper": 4 * u("t/MW"), "silicon": 5 * u("t/MW")}

In [3]:
EIC_wind = 0
EIC_solar = 0

for i, CF_i in CF.items():
    EIC_wind += CF_i * MI_wind.get(i, 0)
    EIC_solar += CF_i * MI_solar.get(i, 0)

print("Emissions intensity of capital:")
print(f"wind: {EIC_wind}")
print(f"solar: {EIC_solar}")

Emissions intensity of capital:
wind: 303.0 metric_ton / megawatt
solar: 566.4 metric_ton / megawatt


What is the **emissions intensity of electricity** for **(c) wind** and **(d) solar** assuming that the above-calculated $\mathrm{EIC}\_{\text{mat}}$ is the dominant cause of emissions over the project lifetime (approximate $\mathrm{EIC}\_{\text{proc}}$, $\mathrm{EIC}\_{\text{proc}}$, and $\mathrm{EIO}$ to all be zero). Use the following lifetimes and capacity factors:

|                | wind | solar |
|----------------|------|-------|
|$L$ / \[years]  | 25   | 30    |
|$C$             | 50%  | 10%   |

In [4]:
E_wind = (25 * u("yr") * 0.50).to("MWh/MW")
print(f"Lifetime electricity from wind: {E_wind}")
E_solar = (30 * u("yr") * 0.10).to("MWh/MW")
print(f"Lifetime electricity from solar: {E_solar}")

Lifetime electricity from wind: 109575.0 megawatt_hour / megawatt
Lifetime electricity from solar: 26298.0 megawatt_hour / megawatt


In [5]:
EIE_wind = (EIC_wind / E_wind).to("g/kWh")
print(f"Emissions intensity of electricity from wind: {EIE_wind}")
EIE_solar = (EIC_solar / E_solar).to("g/kWh")
print(f"Emissions intensity of electricity from wind: {EIE_solar}")

Emissions intensity of electricity from wind: 2.765229295003423 gram / kilowatt_hour
Emissions intensity of electricity from wind: 21.5377595254392 gram / kilowatt_hour


Financial cost example: wind power
----------------------------------
You're considering investing in a small onshore wind farm, with five 3-MW wind turbines (15 MW).

You can buy the turbines for 3 million EUR a piece. They have a lifetime of 25 years. Assume that on-shore wind where you are building gets a capacity factor of 35%. Assume that operating expenditures and capital expenditures are negligible compared to the cost of the turbines.

Use a depriciation rate of 3%.

What is the levelized cost of electricity?

In [6]:
CAPEX = 5 * (3 * 1e6)  # EUR. Unfortunately, pint does not have currency units.
OPEX = 0
P_max = 5 * 3 * u("MW")
C = 0.35
r = 0.03

L = 25

In [7]:
total_present_cost = 0
total_present_energy = 0

for y in range(L):
    print(f"year {y}:")
    if y==0: 
        cost_y = CAPEX
        energy = 0
    else:
        cost_y = OPEX
        energy = P_max * C * 1 * u("yr")
        energy.ito("MWh")
        
    depriciation_factor = (1 + r)**y
    present_cost = cost_y / depriciation_factor
    print(f"present value of cost: {present_cost}")
    present_energy = energy / depriciation_factor
    print(f"present value of energy: {present_energy}")
    total_present_cost += present_cost
    total_present_energy += present_energy

print("------")
print(f"total present value of cost: {total_present_cost}")
print(f"total present value of energy: {total_present_energy}")

LCOE = total_present_cost / total_present_energy
print("------")
print(f"LCOE = EUR {LCOE}")

year 0:
present value of cost: 15000000.0
present value of energy: 0.0
year 1:
present value of cost: 0.0
present value of energy: 44681.06796116505 megawatt_hour
year 2:
present value of cost: 0.0
present value of energy: 43379.67763219908 megawatt_hour
year 3:
present value of cost: 0.0
present value of energy: 42116.19187592143 megawatt_hour
year 4:
present value of cost: 0.0
present value of energy: 40889.50667565187 megawatt_hour
year 5:
present value of cost: 0.0
present value of energy: 39698.550170535804 megawatt_hour
year 6:
present value of cost: 0.0
present value of energy: 38542.28171896679 megawatt_hour
year 7:
present value of cost: 0.0
present value of energy: 37419.690989288145 megawatt_hour
year 8:
present value of cost: 0.0
present value of energy: 36329.797076978786 megawatt_hour
year 9:
present value of cost: 0.0
present value of energy: 35271.647647552214 megawatt_hour
year 10:
present value of cost: 0.0
present value of energy: 34244.318104419624 megawatt_hour
yea